# Feature Engineering Pipeline - Sanika Hajare
Leak-Free ML Pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Dataset

In [ ]:
np.random.seed(42)
n=1000
df = pd.DataFrame({
    'Age': np.random.randint(21,70,n),
    'Income': np.random.normal(50000, 15000, n),
    'CreditScore': np.random.normal(650, 100, n),
    'LoanAmount': np.random.normal(15000, 5000, n),
    'EmploymentYears': np.random.randint(0,30,n),
    'Gender': np.random.choice(['Male','Female'], n),
    'Education': np.random.choice(['Graduate','Undergrad','PhD'], n),
    'Married': np.random.choice(['Yes','No'], n),
    'Default': np.random.choice([0,1], n, p=[0.8,0.2])
})
for col in ['Income','CreditScore','Education']:
    df.loc[df.sample(frac=0.05).index, col] = np.nan
df.head()

## 2. Train/Test Split BEFORE Transformation (Prevent Leakage)

In [ ]:
X = df.drop('Default', axis=1)
y = df['Default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. ColumnTransformer Pipeline

In [ ]:
numeric_features = ['Age','Income','CreditScore','LoanAmount','EmploymentYears']
categorical_features = ['Gender','Education','Married']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
full_pipeline.fit(X_train, y_train)
print(f'Train Score: {full_pipeline.score(X_train, y_train):.3f}')
print(f'Test Score: {full_pipeline.score(X_test, y_test):.3f}')

## 4. Correlation Analysis

In [ ]:
ohe_features = list(full_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_features
X_train_transformed = full_pipeline.named_steps['preprocessor'].transform(X_train)

plt.figure(figsize=(8,4))
sns.heatmap(pd.DataFrame(X_train_transformed, columns=all_features).corr(), cmap='coolwarm')
plt.title('Correlation After Preprocessing')
plt.show()

## 5. Feature Importance

In [ ]:
mi_scores = mutual_info_classif(X_train_transformed, y_train)
mi_df = pd.DataFrame({'Feature': all_features, 'MI_Score': mi_scores}).sort_values('MI_Score', ascending=False)
print(mi_df)

plt.figure(figsize=(8,5))
sns.barplot(data=mi_df, x='MI_Score', y='Feature')
plt.title('Mutual Info Importance')
plt.show()

rf_imp = full_pipeline.named_steps['classifier'].feature_importances_
imp_df = pd.DataFrame({'Feature': all_features, 'RF_Importance': rf_imp}).sort_values('RF_Importance', ascending=False)
print(imp_df)
sns.barplot(data=imp_df, x='RF_Importance', y='Feature')
plt.title('RF Importance')
plt.show()